# Basic RAG Starter

A minimal RAG baseline using minsearch, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Replace the example data with your own and adapt the instructions to match what your project should actually do.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Replace this example with your own data. Each document should be a dict with a few text fields you want to search over.

In [2]:
import os
import duckdb

# ── 1. Create a synthetic star-schema DuckDB database ─────────────────────────
db_path = "../data/sales.duckdb"
os.makedirs("../data", exist_ok=True)

con = duckdb.connect(db_path)
con.execute("DROP TABLE IF EXISTS orders")
con.execute("DROP TABLE IF EXISTS products")
con.execute("DROP TABLE IF EXISTS customers")

con.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        VARCHAR,
    segment     VARCHAR,   -- Enterprise | Mid-Market | SMB
    country     VARCHAR
)""")
con.execute("COMMENT ON TABLE customers IS 'One row per customer.'")

con.execute("""
CREATE TABLE products (
    product_id  INTEGER PRIMARY KEY,
    name        VARCHAR,
    category    VARCHAR,   -- Software | Hardware | Services
    unit_price  DOUBLE
)""")
con.execute("COMMENT ON TABLE products IS 'One row per product.'")

con.execute("""
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    product_id  INTEGER REFERENCES products(product_id),
    order_date  DATE,
    quantity    INTEGER,
    region      VARCHAR    -- North | South | East | West
)""")
con.execute("COMMENT ON TABLE orders IS 'One row per order line. Revenue = quantity * unit_price (join products).'")

con.execute("""
INSERT INTO customers VALUES
  (1,'Acme Corp','Enterprise','US'),(2,'Globex','Mid-Market','UK'),
  (3,'Initech','SMB','DE'),(4,'Umbrella','Enterprise','US'),
  (5,'Hooli','Mid-Market','US'),(6,'Dunder Mifflin','SMB','US'),
  (7,'Pied Piper','SMB','US'),(8,'Massive Dynamics','Enterprise','UK'),
  (9,'Cyberdyne','Enterprise','US'),(10,'Soylent Corp','Mid-Market','CA')
""")

con.execute("""
INSERT INTO products VALUES
  (1,'Analytics Suite','Software',1200.00),(2,'Data Connector','Software',450.00),
  (3,'Server Rack','Hardware',3500.00),(4,'Laptop Pro','Hardware',1800.00),
  (5,'Consulting Day','Services',2000.00),(6,'Training Pack','Services',800.00),
  (7,'Cloud Gateway','Software',600.00),(8,'NAS Device','Hardware',1100.00)
""")

con.execute("""
INSERT INTO orders VALUES
  (1,1,1,'2024-01-15',3,'North'),(2,1,5,'2024-02-10',2,'North'),
  (3,2,2,'2024-01-20',5,'East'), (4,2,4,'2024-03-05',1,'East'),
  (5,3,6,'2024-02-28',2,'South'),(6,4,1,'2024-01-30',10,'West'),
  (7,4,3,'2024-04-12',2,'West'), (8,5,7,'2024-03-18',4,'North'),
  (9,6,6,'2024-04-01',3,'South'),(10,7,2,'2024-04-22',6,'East'),
  (11,8,1,'2024-05-03',5,'East'), (12,8,5,'2024-05-10',3,'East'),
  (13,9,3,'2024-02-14',1,'West'), (14,9,4,'2024-06-01',4,'West'),
  (15,10,7,'2024-06-15',8,'North'),(16,1,8,'2024-06-20',2,'North'),
  (17,3,1,'2024-07-04',1,'South'),(18,5,5,'2024-07-11',2,'North'),
  (19,6,4,'2024-08-03',3,'South'),(20,2,8,'2024-08-19',4,'East')
""")

con.close()
print("Database created at", db_path)

Database created at ../data/sales.duckdb


In [3]:
# ── Function to generate schema documents ─────────────────────────
def generate_schema_documents(db_path):
    con = duckdb.connect(db_path, read_only=True)
    
    tables = ['customers', 'products', 'orders']
    
    documents = []
    
    for table in tables:
        # Get table comment
        table_comment_df = con.execute(f"SELECT comment FROM duckdb_tables() WHERE table_name = '{table}'").df()
        table_comment = table_comment_df.iloc[0]['comment'] if not table_comment_df.empty else ""
        
        # Get columns
        columns_df = con.execute(f"SELECT column_name, data_type FROM duckdb_columns() WHERE table_name = '{table}' ORDER BY column_index").df()
        
        col_strs = []
        for _, row in columns_df.iterrows():
            col_name = row['column_name']
            col_type = row['data_type']
            
            # Check if PK
            pk_query = f"""
            SELECT 1 FROM duckdb_constraints() 
            WHERE table_name = '{table}' AND constraint_type = 'PRIMARY KEY' 
            AND '{col_name}' = ANY(constraint_column_names)
            """
            pk_df = con.execute(pk_query).df()
            if not pk_df.empty:
                col_type += ' PK'
            
            # Check FK
            fk_query = f"""
            SELECT referenced_table FROM duckdb_constraints() 
            WHERE table_name = '{table}' AND constraint_type = 'FOREIGN KEY' 
            AND '{col_name}' = ANY(constraint_column_names)
            """
            fk_df = con.execute(fk_query).df()
            if not fk_df.empty:
                ref_table = fk_df.iloc[0]['referenced_table']
                col_type += f' FK→{ref_table}'
            
            col_strs.append(f"{col_name} {col_type}")
        
        schema_str = f"{table}({', '.join(col_strs)}). "
        
        # Get distinct values for categorical columns
        desc_parts = [table_comment]
        if table == 'customers':
            segments = con.execute("SELECT DISTINCT segment FROM customers ORDER BY segment").df()['segment'].tolist()
            desc_parts.append(f"segment is one of: {', '.join(segments)}.")
        elif table == 'products':
            categories = con.execute("SELECT DISTINCT category FROM products ORDER BY category").df()['category'].tolist()
            desc_parts.append(f"category is one of: {', '.join(categories)}.")
        elif table == 'orders':
            regions = con.execute("SELECT DISTINCT region FROM orders ORDER BY region").df()['region'].tolist()
            desc_parts.append(f"region is one of: {', '.join(regions)}.")
        
        full_desc = schema_str + ' '.join(desc_parts)
        
        documents.append({
            "type": "schema",
            "table_name": table,
            "description": full_desc,
            "question": "",
            "sql": "",
        })
    
    con.close()
    return documents

schema_docs = generate_schema_documents(db_path)

In [4]:
schema_docs

[{'type': 'schema',
  'table_name': 'customers',
  'description': 'customers(customer_id INTEGER PK, name VARCHAR, segment VARCHAR, country VARCHAR). One row per customer. segment is one of: Enterprise, Mid-Market, SMB.',
  'question': '',
  'sql': ''},
 {'type': 'schema',
  'table_name': 'products',
  'description': 'products(product_id INTEGER PK, name VARCHAR, category VARCHAR, unit_price DOUBLE). One row per product. category is one of: Hardware, Services, Software.',
  'question': '',
  'sql': ''},
 {'type': 'schema',
  'table_name': 'orders',
  'description': 'orders(order_id INTEGER PK, customer_id INTEGER FK→customers, product_id INTEGER FK→products, order_date DATE, quantity INTEGER, region VARCHAR). One row per order line. Revenue = quantity * unit_price (join products). region is one of: East, North, South, West.',
  'question': '',
  'sql': ''}]

In [5]:
# ── 2. Define the RAG corpus: schema descriptions + example NL→SQL pairs ──────
examples = [
    # --- Example NL→SQL pairs ---
    {
        "type": "example",
        "table_name": "orders",
        "description": "Total revenue per region, sorted highest first",
        "question": "What is the total revenue by region?",
        "sql": (
            "SELECT region, ROUND(SUM(o.quantity * p.unit_price), 2) AS total_revenue "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY region ORDER BY total_revenue DESC"
        ),
    },
    {
        "type": "example",
        "table_name": "customers",
        "description": "Top customers ranked by total spend",
        "question": "Who are the top 5 customers by total revenue?",
        "sql": (
            "SELECT c.name, ROUND(SUM(o.quantity * p.unit_price), 2) AS total_spend "
            "FROM orders o "
            "JOIN customers c USING (customer_id) "
            "JOIN products p USING (product_id) "
            "GROUP BY c.name ORDER BY total_spend DESC LIMIT 5"
        ),
    },
    {
        "type": "example",
        "table_name": "orders",
        "description": "Monthly revenue trend across all regions",
        "question": "What is the monthly revenue trend?",
        "sql": (
            "SELECT strftime(order_date, '%Y-%m') AS month, "
            "ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY month ORDER BY month"
        ),
    },
    {
        "type": "example",
        "table_name": "products",
        "description": "Revenue breakdown by product category",
        "question": "Which product category generates the most revenue?",
        "sql": (
            "SELECT p.category, ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY p.category ORDER BY revenue DESC"
        ),
    },
    {
        "type": "example",
        "table_name": "customers",
        "description": "Revenue by customer segment (Enterprise, Mid-Market, SMB)",
        "question": "How does revenue compare across customer segments?",
        "sql": (
            "SELECT c.segment, ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue "
            "FROM orders o "
            "JOIN customers c USING (customer_id) "
            "JOIN products p USING (product_id) "
            "GROUP BY c.segment ORDER BY revenue DESC"
        ),
    },
    {
        "type": "example",
        "table_name": "orders",
        "description": "Number of orders and total quantity sold per product",
        "question": "How many orders does each product have?",
        "sql": (
            "SELECT p.name, COUNT(*) AS num_orders, SUM(o.quantity) AS total_qty "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY p.name ORDER BY num_orders DESC"
        ),
    },
]

documents = schema_docs + examples

print(f"Loaded {len(documents)} documents into corpus "
      f"({sum(1 for d in documents if d['type']=='schema')} schema, "
      f"{sum(1 for d in documents if d['type']=='example')} examples)")

Loaded 9 documents into corpus (3 schema, 6 examples)


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [6]:
index = Index(
    text_fields=["description", "question", "sql"],
    keyword_fields=["type", "table_name"],
)

index.fit(documents)
print("Index ready —", len(documents), "documents indexed")


Index ready — 9 documents indexed


## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [7]:
def search(query):
    return index.search(query, num_results=5)

In [8]:
instructions = """
You are a DuckDB SQL expert. Your job is to turn a business user's natural-language
question into a single, valid DuckDB SQL query.

Rules:
- Use ONLY the tables and columns described in the SCHEMA entries in CONTEXT.
- Prefer the patterns shown in the EXAMPLE entries in CONTEXT when they are relevant.
- Revenue is always quantity * unit_price (requires joining orders with products).
- Return ONLY a JSON object with two keys, no markdown fences:
    {"sql": "<the SQL query>", "explanation": "<one sentence plain-English explanation>"}
"""


def build_prompt(query, search_results):
    schema_entries = [r for r in search_results if r.get("type") == "schema"]
    example_entries = [r for r in search_results if r.get("type") == "example"]

    schema_block = "\n".join(
        f"- {r['description']}" for r in schema_entries
    ) or "(no schema retrieved)"

    example_block = "\n".join(
        f"- Q: {r['question']}\n  SQL: {r['sql']}"
        for r in example_entries
        if r.get("question")
    ) or "(no examples retrieved)"

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
SCHEMA:
{schema_block}

EXAMPLES:
{example_block}
</CONTEXT>
""".strip()

    return user_prompt


In [9]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [12]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    raw = llm(prompt, instructions)

    # Parse the JSON the LLM returns
    try:
        parsed = json.loads(raw)
        sql = parsed.get("sql", "").strip()
        explanation = parsed.get("explanation", "")
    except json.JSONDecodeError:
        # Fallback: treat raw response as SQL if it isn't valid JSON
        sql = raw.strip()
        explanation = ""

    # Execute SQL against DuckDB
    con = duckdb.connect("../data/sales.duckdb", read_only=True)
    try:
        results = con.execute(sql).df().to_dict(orient="records")
        error = None
    except Exception as e:
        results = []
        error = str(e)
    finally:
        con.close()

    return {
        "query":       query,
        "sql":         sql,
        "explanation": explanation,
        "results":     results,
        "error":       error,
        "trust": {
            "schema_docs_used":  [r["table_name"] for r in search_results if r.get("type") == "schema"],
            "examples_retrieved": sum(1 for r in search_results if r.get("type") == "example"),
        },
    }


## 4. Try It Out

In [13]:
result = rag("Which region had the highest revenue, and how does each region compare?")

print("Query      :", result["query"])
print("SQL        :", result["sql"])
print("Explanation:", result["explanation"])
print("Trust      :", result["trust"])
print("Error      :", result["error"])
print("\nResults:")
for row in result["results"]:
    print(" ", row)


Query      : Which region had the highest revenue, and how does each region compare?
SQL        : SELECT region, ROUND(SUM(o.quantity * p.unit_price), 2) AS total_revenue FROM orders o JOIN products p USING (product_id) GROUP BY region ORDER BY total_revenue DESC
Explanation: This query calculates total revenue for each region and orders the results to identify which region had the highest revenue.
Trust      : {'schema_docs_used': [], 'examples_retrieved': 5}
Error      : None

Results:
  {'region': 'West', 'total_revenue': 29700.0}
  {'region': 'East', 'total_revenue': 23150.0}
  {'region': 'North', 'total_revenue': 21000.0}
  {'region': 'South', 'total_revenue': 10600.0}


In [14]:
rag("What was the biggest sale?")

{'query': 'What was the biggest sale?',
 'sql': 'SELECT MAX(o.quantity * p.unit_price) AS biggest_sale FROM orders o JOIN products p USING (product_id)',
 'explanation': 'This query calculates the maximum sale amount by multiplying quantity with unit price for each order.',
 'results': [{'biggest_sale': 12000.0}],
 'error': None,
 'trust': {'schema_docs_used': [], 'examples_retrieved': 4}}